# htmx v4 version of `counter-rsjs.py`

In [1]:
from fasthtml.common import *
from fasthtml.jupyter import *
from uuid import uuid4
from string import Formatter

def extract_names(s): return [o for _,o,_,_ in Formatter().parse(s) if o is not None]

class RsJs:
    def __init__(self, nm): self.nm = nm
    def ref(self, k): return f'me("[{self.data(k)}]", el)'
    def expand(self, s):
        d = {o:self.ref(o) for o in extract_names(s)}
        return s.format(**d)
    def data(self, nm=''):
        if not nm: return f'data-{self.nm}'
        return f'data-{self.nm}-{nm}'
    def __getattr__(self, k): return {self.data(k): True}
    @property
    def d(self): return {self.data(): True}
    def __call__(self, code):
        return Script(f'proc_htmx("[{self.data()}]", el => {{ {self.expand(code)} }})')

app, rt = fast_app(htmx=False, htmx4=True)

r = RsJs('incrementer')

def Incrementer(start=0, btn_txt='Increment'):
    return Section(r.d)(
        Output(start, r.output, id='my-output'),
        Button(btn_txt, r.increment)
    )

@rt("/")
def get():
    return Titled('RSJS Incrementer',
        r("htmx.on({increment}, 'click', _=>{output}.value++)"),
        Incrementer(),
        Incrementer(5, 'Do it')
    )

srv = JupyUvi(app)